---

# Dudley Disability Data Cleaning

---

## Purpose

Clean and validate Census 2021 disability data for Dudley residents aged
16 years and over, ready for later analysis of local employment barriers
and possible support needs.

The source is RM019 — Economic activity status by hours worked by
disability, accessed through ONS/Nomis. The economic activity selection
is Total, covering all economic activity statuses.

## Objectives

- Inspect the raw file structure and load the dataset.
- Identify and handle blank rows and other non-data rows.
- Check missing values, duplicates and LSOA identifiers.
- Standardise column names and validate numeric counts.
- Save a cleaned CSV and document the cleaning decisions.

## Inputs

- `raw_datasets/dudley_disability_16_plus.csv`
- Census 2021 data downloaded from Nomis on 16 October 2026.

## Outputs

- `cleaned_datasets/dudley_disability_16_cleaned.csv`
- Documented checks, findings and a cleaning conclusion.

## Ethical Considerations

Disability status does not establish a person's ability or willingness
to work, or their individual support needs. These neighbourhood statistics
must not be used to make assumptions about individual residents.

The population includes everyone aged 16 and over, including older
residents. It must not be described as exclusively working age.

## Additional Comments

The raw source file will remain unchanged during cleaning.
Percentage calculations, analysis, visualisations and modelling will
be completed in separate notebooks.

Code developed with assistance from ChatGPT will be acknowledged,
with relevant documentation linked in comments.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

### Import libraries

Import pandas to load, inspect and clean the disability dataset.

In [4]:
# Import pandas for loading and cleaning the dataset.
import pandas as pd

## 1. Inspect the raw file structure

Read the first ten lines to check the dataset title, population and
column-heading row before loading the CSV with pandas.

In [5]:
# Developed with assistance from ChatGPT.
raw_path = "raw_datasets/dudley_disability_16_plus.csv"

# Read the source headings without changing the raw file.
with open(raw_path, "r", encoding="utf-8-sig") as file:
    for line_number in range(1, 11):
        line = file.readline()
        if not line:
            break
        print(f"{line_number}: {line.rstrip()}")

1: RM019 - Economic activity status by hours worked by disability,,,,,,
2: ONS Crown Copyright Reserved [from Nomis on 16 October 2026],,,,,,
3: Population :,All usual residents aged 16 years and over,,,,,
4: Units      :,Persons,,,,,
5: Date       :,2021,,,,,
6: Economic activity status and hours worked:,Total,,,,,
7: ,,,,,,
8: 2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
9: ,,,,,,
10: Dudley 001A,E01009763,1640,350,154,196,1290


### Findings

- The file contains Census 2021 RM019 disability data, downloaded
  from Nomis on 16 October 2026.
- The population is all usual residents aged 16 years and over.
  Counts are measured in persons.
- The economic activity selection is Total, so the data includes
  all economic activity statuses.
- Column headings appear on line 8. Skip the first seven lines
  when loading the CSV.
- Line 9 is an empty record containing commas. This will be
  checked during cleaning.
- The table contains LSOA names, codes and five count columns.
- The two categories “limited a lot” and “limited a little” are
  subcategories of the disabled total. They must not be added
  to that total, as this would double-count residents.

---

## 2. Load and preview the dataset

Load the CSV, skipping the seven introductory lines so that line 8
provides the column headings. Preview five rows and check the initial
number of rows and columns before cleaning.

In [6]:
# Developed with assistance from ChatGPT.
# Load the data using the column headings identified during inspection.
disability_df = pd.read_csv(
    raw_path,
    skiprows=7,
    encoding="utf-8-sig"
)

# Check the initial dimensions and preview five rows.
print(f"Rows: {disability_df.shape[0]}")
print(f"Columns: {disability_df.shape[1]}")
disability_df.head()

Rows: 211
Columns: 7


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dudley 001A,E01009763,1640.0,350.0,154.0,196.0,1290.0
2,Dudley 001B,E01009764,1264.0,261.0,121.0,140.0,1003.0
3,Dudley 001C,E01009767,1268.0,296.0,145.0,151.0,972.0
4,Dudley 001D,E01009769,1428.0,376.0,208.0,168.0,1052.0


### Findings

- The initial dataset contains 211 rows and 7 columns.
- The column headings have loaded correctly.
- The first row contains missing values in every column, matching
  the empty record identified in the raw file.
- The following rows contain Dudley LSOA names, codes and counts.
- Counts currently display with decimal places, such as 1640.0.
  Numeric types will be checked during cleaning.
- The 211 rows are not yet a verified count of neighbourhoods.
  Blank rows and any source notes must be inspected before removal.

---

## 3. Inspect blank rows and rows without area codes

Count completely empty rows and inspect any rows without a mnemonic
(LSOA code). This helps identify blank records and source notes before
deciding which rows to remove.

In [7]:
# Developed with assistance from ChatGPT.

# Identify rows where every value is missing.
blank_rows = disability_df.isna().all(axis=1)

# Identify missing or whitespace-only area codes.
missing_codes = (
    disability_df["mnemonic"].isna()
    | disability_df["mnemonic"].astype(str).str.strip().eq("")
)

print(f"Completely empty rows: {blank_rows.sum()}")
print(f"Rows without an area code: {missing_codes.sum()}")

# Show these rows for inspection before removing anything.
disability_df.loc[missing_codes]

Completely empty rows: 4
Rows without an area code: 4


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Findings

- Four rows are completely empty: indices 0, 204, 205 and 206.
- These are also the only four rows without an area code.
- These blank records can be removed without losing population data.
- Removing them would leave 207 rows. These must still be checked
  to distinguish valid Dudley LSOAs from any other records.
- Source notes may contain text in the mnemonic column, so checking
  missing codes alone does not identify every possible non-data row.

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
